In [1]:
# ============================================================
# CROSS-DATASET VALIDATION & RECOVERY
# ============================================================

import pandas as pd
import numpy as np
import os


# ============================================================
# STEP 1 - LOAD CLEANED DATASETS
# ============================================================

BASE_DIR = os.getcwd()
INPUT_DIR = os.path.join(BASE_DIR, "cleaned")
OUTPUT_DIR = os.path.join(BASE_DIR, "cross_dataset_final")

os.makedirs(OUTPUT_DIR, exist_ok=True)

MASTER = os.path.join(INPUT_DIR, "mandi_master_clean.csv")
ARRIVALS = os.path.join(INPUT_DIR, "arrivals_clean.csv")
PRICE = os.path.join(INPUT_DIR, "price_and_msp_clean.csv")
TRANSPORT = os.path.join(INPUT_DIR, "transport_logistics_clean.csv")
WEATHER = os.path.join(INPUT_DIR, "weather_sensors_clean.csv")

print("Input folder :", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)

master = pd.read_csv(MASTER)
arrivals = pd.read_csv(ARRIVALS)
price = pd.read_csv(PRICE)
transport = pd.read_csv(TRANSPORT)
weather = pd.read_csv(WEATHER)

print("All cleaned datasets loaded.\n")

print("Master   :", master.shape)
print("Arrivals :", arrivals.shape)
print("Price    :", price.shape)
print("Transport:", transport.shape)
print("Weather  :", weather.shape)


# ============================================================
# STEP 2 - MASTER VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("MASTER VALIDATION")
print("=" * 70)

print("Rows:", len(master))
print("Unique mandi IDs:", master["mandi_id"].nunique())
print("Duplicate mandi IDs:", master["mandi_id"].duplicated().sum())
print("Missing mandi IDs:", master["mandi_id"].isna().sum())

assert master["mandi_id"].is_unique
assert master["mandi_id"].notna().all()

print("PASS: Master mandi_id is unique and complete.")


# ============================================================
# STEP 3 - CHECK JOINS
# ============================================================

master_ids = set(
    master["mandi_id"]
    .dropna()
    .astype(str)
)

print("\n" + "=" * 70)
print("REFERENTIAL INTEGRITY")
print("=" * 70)

for name, df in {
    "Arrivals": arrivals,
    "Price": price,
    "Transport": transport
}.items():

    dataset_ids = set(
        df["mandi_id"]
        .dropna()
        .astype(str)
    )

    invalid = dataset_ids - master_ids

    print(f"\n{name}")
    print("Unique mandi IDs:", len(dataset_ids))
    print("Invalid mandi IDs:", len(invalid))

    if len(invalid) == 0:
        print("PASS")
    else:
        print("WARNING:", list(invalid)[:10])


# ============================================================
# STEP 4 - ARRIVALS ENRICHMENT
# ============================================================

print("\n" + "=" * 70)
print("ARRIVALS -> MASTER ENRICHMENT")
print("=" * 70)

arrival_rows_before = len(arrivals)

arrival_lookup = master[
    [
        "mandi_id",
        "mandi_name",
        "district",
        "state",
        "mandi_type"
    ]
].copy()

arrivals = arrivals.merge(
    arrival_lookup,
    on="mandi_id",
    how="left"
)

print("Rows before:", arrival_rows_before)
print("Rows after :", len(arrivals))

assert len(arrivals) == arrival_rows_before

print("\nAdded columns:")
print("mandi_name")
print("district")
print("state")
print("mandi_type")

print("\nMissing values in added columns:")
print(
    arrivals[
        [
            "mandi_name",
            "district",
            "state",
            "mandi_type"
        ]
    ].isna().sum()
)

print("\nPASS: Arrivals successfully enriched from Master.")


# ============================================================
# STEP 5 - PRICE DISTRICT RECOVERY
# ============================================================

print("\n" + "=" * 70)
print("PRICE DISTRICT RECOVERY")
print("=" * 70)

# Existing district_final came from previous cleaning.
# We only fill it when Master has a district.

before = price["district_final"].isna().sum()

print("Missing district_final BEFORE:", before)

district_lookup = master[
    ["mandi_id", "district"]
].copy()

district_lookup = district_lookup.rename(
    columns={
        "district": "master_district"
    }
)

price = price.merge(
    district_lookup,
    on="mandi_id",
    how="left"
)

recoverable = (
    price["district_final"].isna()
    &
    price["master_district"].notna()
)

recovered = recoverable.sum()

price.loc[
    recoverable,
    "district_final"
] = price.loc[
    recoverable,
    "master_district"
]

after = price["district_final"].isna().sum()

print("Recoverable from Master:", recovered)
print("Missing district_final AFTER:", after)

print(
    "Validation:",
    before - recovered == after
)

assert before - recovered == after


# ============================================================
# STEP 6 - MSP VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("MSP VALIDATION")
print("=" * 70)

# MSP was already recovered during the price cleaning stage.

msp_missing = price["msp"].isna().sum()

print("Missing MSP:", msp_missing)

if "msp_was_imputed" in price.columns:

    print(
        "Previously imputed MSP:",
        price["msp_was_imputed"].sum()
    )

print(
    "\nRemaining MSP values are not filled again."
)

print(
    "Reason: the previous cleaning stage already "
    "recovered all MSP values that had a valid "
    "crop_name + date lookup."
)


# ============================================================
# STEP 7 - TRANSPORT ENRICHMENT
# ============================================================

print("\n" + "=" * 70)
print("TRANSPORT -> MASTER ENRICHMENT")
print("=" * 70)

transport_rows_before = len(transport)

transport_lookup = master[
    [
        "mandi_id",
        "mandi_name",
        "district",
        "state",
        "mandi_type"
    ]
].copy()

transport = transport.merge(
    transport_lookup,
    on="mandi_id",
    how="left"
)

print("Rows before:", transport_rows_before)
print("Rows after :", len(transport))

assert len(transport) == transport_rows_before

print("\nMissing Master attributes:")
print(
    transport[
        [
            "mandi_name",
            "district",
            "state",
            "mandi_type"
        ]
    ].isna().sum()
)

print("\nPASS: Transport successfully enriched from Master.")


# ============================================================
# STEP 8 - TRANSPORT TRANSIT VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("TRANSPORT TRANSIT VALIDATION")
print("=" * 70)

print(
    "Missing transit_hours_final:",
    transport["transit_hours_final"].isna().sum()
)

print(
    "Negative transit_hours_final:",
    (transport["transit_hours_final"] < 0).sum()
)

print(
    "Previously imputed:",
    transport["transit_hours_imputed"].sum()
)

print(
    "Average transit hours:",
    round(
        transport["transit_hours_final"].mean(),
        2
    )
)

print(
    "Median transit hours:",
    round(
        transport["transit_hours_final"].median(),
        2
    )
)

print(
    "Maximum transit hours:",
    round(
        transport["transit_hours_final"].max(),
        2
    )
)


# ============================================================
# STEP 9 - WEATHER VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("WEATHER VALIDATION")
print("=" * 70)

print("Rows:", len(weather))
print(
    "Unique sensors:",
    weather["sensor_id"].nunique()
)

print(
    "Missing temperature:",
    weather["temperature_c"].isna().sum()
)

print(
    "Missing rainfall:",
    weather["rainfall_mm"].isna().sum()
)

print(
    "Missing humidity:",
    weather["humidity_percent"].isna().sum()
)

print(
    "\nNo sensor measurements are artificially filled "
    "from another dataset."
)


# ============================================================
# STEP 10 - WEATHER DAILY AGGREGATION
# ============================================================

print("\n" + "=" * 70)
print("WEATHER DAILY AGGREGATION")
print("=" * 70)

weather["date"] = pd.to_datetime(
    weather["timestamp_ist"],
    errors="coerce"
).dt.date

weather_daily = (
    weather
    .groupby("date")
    .agg(
        avg_temperature_c=("temperature_c", "mean"),
        total_rainfall_mm=("rainfall_mm", "sum"),
        avg_humidity=("humidity_percent", "mean"),
        sensor_count=("sensor_id", "nunique")
    )
    .reset_index()
)

print("Weather daily shape:", weather_daily.shape)

print("\nFirst 5 rows:")
display(weather_daily.head())


# ============================================================
# STEP 11 - CLEAN TEMPORARY PRICE COLUMN
# ============================================================

price.drop(
    columns=["master_district"],
    errors="ignore",
    inplace=True
)


# ============================================================
# STEP 12 - RECOVERY AUDIT
# ============================================================

audit = pd.DataFrame({

    "dataset": [
        "Price",
        "Price",
        "Arrivals",
        "Transport",
        "Weather"
    ],

    "operation": [
        "District recovery",
        "MSP validation",
        "Master enrichment",
        "Master enrichment",
        "Sensor validation"
    ],

    "source": [
        "Mandi Master",
        "Previous cleaning stage",
        "Mandi Master",
        "Mandi Master",
        "No cross-dataset source"
    ],

    "matching_key": [
        "mandi_id",
        "crop_name + date",
        "mandi_id",
        "mandi_id",
        "None"
    ],

    "missing_before": [
        before,
        msp_missing,
        np.nan,
        np.nan,
        np.nan
    ],

    "values_recovered": [
        recovered,
        0,
        np.nan,
        np.nan,
        0
    ],

    "missing_after": [
        after,
        msp_missing,
        np.nan,
        np.nan,
        np.nan
    ]
})

print("\n" + "=" * 70)
print("CROSS-DATASET AUDIT")
print("=" * 70)

display(audit)


# ============================================================
# STEP 13 - FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("\nMASTER")
print("Rows:", len(master))
print("Unique mandi IDs:", master["mandi_id"].nunique())

print("\nARRIVALS")
print("Rows:", len(arrivals))
print("Unique mandi IDs:", arrivals["mandi_id"].nunique())
print(
    "Unmatched mandi names:",
    arrivals["mandi_name"].isna().sum()
)

print("\nPRICE")
print("Rows:", len(price))
print(
    "Missing district_final:",
    price["district_final"].isna().sum()
)
print(
    "Missing MSP:",
    price["msp"].isna().sum()
)

print("\nTRANSPORT")
print("Rows:", len(transport))
print(
    "Missing transit_hours_final:",
    transport["transit_hours_final"].isna().sum()
)
print(
    "Negative transit_hours_final:",
    (
        transport["transit_hours_final"] < 0
    ).sum()
)
print(
    "Unmatched mandi names:",
    transport["mandi_name"].isna().sum()
)

print("\nWEATHER")
print("Rows:", len(weather))
print(
    "Unique sensors:",
    weather["sensor_id"].nunique()
)
print(
    "Missing temperature:",
    weather["temperature_c"].isna().sum()
)
print(
    "Missing rainfall:",
    weather["rainfall_mm"].isna().sum()
)

print("\nValidation complete.")


# ============================================================
# STEP 14 - SAVE FINAL DATASETS
# ============================================================

# OUTPUT_DIR is defined at the top as currentdir/cross_dataset_final.

arrivals.to_csv(
    f"{OUTPUT_DIR}/arrivals_final.csv",
    index=False
)

price.to_csv(
    f"{OUTPUT_DIR}/price_and_msp_final.csv",
    index=False
)

transport.to_csv(
    f"{OUTPUT_DIR}/transport_logistics_final.csv",
    index=False
)

weather.to_csv(
    f"{OUTPUT_DIR}/weather_sensors_final.csv",
    index=False
)

weather_daily.to_csv(
    f"{OUTPUT_DIR}/weather_daily.csv",
    index=False
)

audit.to_csv(
    f"{OUTPUT_DIR}/cross_dataset_recovery_audit.csv",
    index=False
)

print("\nAll final datasets saved successfully.")

Input folder : d:\datathone real\cleaned
Output folder: d:\datathone real\cross_dataset_final
All cleaned datasets loaded.

Master   : (57, 6)
Arrivals : (23767, 11)
Price    : (12000, 13)
Transport: (10000, 18)
Weather  : (15000, 16)

MASTER VALIDATION
Rows: 57
Unique mandi IDs: 57
Duplicate mandi IDs: 0
Missing mandi IDs: 0
PASS: Master mandi_id is unique and complete.

REFERENTIAL INTEGRITY

Arrivals
Unique mandi IDs: 57
Invalid mandi IDs: 0
PASS

Price
Unique mandi IDs: 57
Invalid mandi IDs: 0
PASS

Transport
Unique mandi IDs: 57
Invalid mandi IDs: 0
PASS

ARRIVALS -> MASTER ENRICHMENT
Rows before: 23767
Rows after : 23767

Added columns:
mandi_name
district
state
mandi_type

Missing values in added columns:
mandi_name       0
district      1660
state         1666
mandi_type       0
dtype: int64

PASS: Arrivals successfully enriched from Master.

PRICE DISTRICT RECOVERY
Missing district_final BEFORE: 271
Recoverable from Master: 0
Missing district_final AFTER: 271
Validation: True


,date,avg_temperature_c,total_rainfall_mm,avg_humidity,sensor_count
0,2026-01-01,27.891607,1029.38,62.795455,36
1,2026-01-02,27.595806,1256.93,65.436364,37
2,2026-01-03,25.007000,1242.64,59.000000,36
3,2026-01-04,29.244737,1175.51,66.288462,36
4,2026-01-05,28.708421,1205.95,61.396226,28



CROSS-DATASET AUDIT


,dataset,operation,source,matching_key,missing_before,values_recovered,missing_after
0,Price,District recovery,Mandi Master,mandi_id,271.0,0.0,271.0
1,Price,MSP validation,Previous cleaning stage,crop_name + date,30.0,0.0,30.0
2,Arrivals,Master enrichment,Mandi Master,mandi_id,NaN,NaN,NaN
3,Transport,Master enrichment,Mandi Master,mandi_id,NaN,NaN,NaN
4,Weather,Sensor validation,No cross-dataset source,None,NaN,0.0,NaN



FINAL VALIDATION

MASTER
Rows: 57
Unique mandi IDs: 57

ARRIVALS
Rows: 23767
Unique mandi IDs: 57
Unmatched mandi names: 0

PRICE
Rows: 12000
Missing district_final: 271
Missing MSP: 30

TRANSPORT
Rows: 10000
Missing transit_hours_final: 141
Negative transit_hours_final: 0
Unmatched mandi names: 0

WEATHER
Rows: 15000
Unique sensors: 51
Missing temperature: 0
Missing rainfall: 2309

Validation complete.

All final datasets saved successfully.


In [2]:
print("HELLO WORLD!")

HELLO WORLD!
